In [1]:
from pathlib import Path
from torch import multiprocessing
import os
import torch
from tqdm import tqdm
import soundfile as sf

In [2]:
import numpy as np
import pandas as pd
import time
import io

In [3]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import colors
import datetime as dt

In [4]:
import sys

# append the path of the
# parent directory
sys.path.append('..')
sys.path.append('../src/')
sys.path.append('../src/models/bat_call_detector/batdetect2/')

import src.batdt2_pipeline as batdetect2_pipeline
from pipeline import pipeline
from utils.utils import gen_empty_df
from cfg import get_config
from bat_detect.detector import models
import bat_detect.utils.detector_utils as du
import bat_detect.detector.compute_features as feats
import bat_detect.detector.post_process as pp
import bat_detect.utils.audio_utils as au

In [5]:
def run_models(file_mappings):
    """
    Runs the batdetect2 model to detect bat search-phase calls in the provided audio segments and saves detections into a .csv.

    Parameters
    ------------
    file_mappings : `List`
        - List of dictionaries generated by initialize_mappings()

    Returns
    ------------
    bd_dets : `pandas.DataFrame`
        - A DataFrame of detections that will also be saved in the provided output_dir under the above csv_name
        - 7 columns in this DataFrame: start_time, end_time, low_freq, high_freq, detection_confidence, event, input_file
        - Detections are always specified w.r.t their input_file; earliest start_time can be 0 and latest end_time can be 1795.
        - Events are always "Echolocation" as we are using a model that only detects search-phase calls.
    """

    bd_dets = pd.DataFrame()
    for i in tqdm(range(len(file_mappings))):
        cur_seg = file_mappings[i]
        bd_annotations_df = cur_seg['model']._run_batdetect(cur_seg['audio_seg']['audio_file'])
        bd_offsetted = pipeline._correct_annotation_offsets(
                bd_annotations_df,
                cur_seg['original_file_name'],
                cur_seg['audio_seg']['offset']
            )
        bd_dets = pd.concat([bd_dets, bd_offsetted])
        
    return bd_dets

def apply_models(file_path_mappings, cfg):
    """
    Runs the batdetect2 model to detect bat search-phase calls in the provided audio segments and saves detections into a dataframe

    Parameters
    ------------
    file_mappings : `List`
        - List of dictionaries generated by initialize_mappings()
    cfg : `dict`
        - A dictionary of pipeline parameters:
        - models is the models in the pipeline that are being used.

    Returns
    ------------
    bd_preds : `pandas.DataFrame`
        - A DataFrame of detections that will also be saved in the provided output_dir under the above csv_name
        - 7 columns in this DataFrame: start_time, end_time, low_freq, high_freq, detection_confidence, event, input_file
        - Detections are always specified w.r.t their input_file; earliest start_time can be 0 and latest end_time can be 1795.
        - Events are always "Echolocation" as we are using a model that only detects search-phase calls.
    """

    process_pool = multiprocessing.Pool(cfg['num_processes'])

    bd_dets = tqdm(
            process_pool.imap(apply_model, file_path_mappings, chunksize=1), 
            desc=f"Applying BatDetect2",
            total=len(file_path_mappings),
        )
    
    bd_preds = gen_empty_df() 
    bd_preds = pd.concat(bd_dets, ignore_index=True)
    
    return bd_preds

def apply_model(file_mapping):
    """
    Runs the batdetect2 model on a single provided audio segmens and corrects the offsets according the segment.

    Parameters
    ------------
    file_mappings : `List`
        - List of dictionaries generated by initialize_mappings()

    Returns
    ------------
    corrected_bd_dets : `pandas.DataFrame`
        - A DataFrame of detections that will also be saved in the provided output_dir under the above csv_name
        - 7 columns in this DataFrame: start_time, end_time, low_freq, high_freq, detection_confidence, event, input_file
        - Detections are always specified w.r.t their input_file; earliest start_time can be 0 and latest end_time can be 1795.
        - Events are always "Echolocation" as we are using a model that only detects search-phase calls.
    """

    bd_dets = file_mapping['model']._run_batdetect(file_mapping['audio_seg']['audio_file'])
    corrected_bd_dets = pipeline._correct_annotation_offsets(
                                                            bd_dets,
                                                            file_mapping['original_file_name'],
                                                            file_mapping['audio_seg']['offset']
                                                            )

    return corrected_bd_dets

In [6]:
def run_pipeline_on_file(file, cfg):
    bd_preds = pd.DataFrame()

    if not cfg['output_dir'].is_dir():
        cfg['output_dir'].mkdir(parents=True, exist_ok=True)
    if not cfg['tmp_dir'].is_dir():
        cfg['tmp_dir'].mkdir(parents=True, exist_ok=True)

    cfg["csv_filename"] = f"batdetect2_pipeline_{file.name.split('.')[0]}"
    print(f"Generating detections for {file.name}")
    segmented_file_paths = batdetect2_pipeline.generate_segmented_paths([file], cfg)
    file_path_mappings = batdetect2_pipeline.initialize_mappings(segmented_file_paths, cfg)
    bd_preds = run_models(file_path_mappings)
    if cfg['save']:
        batdetect2_pipeline._save_predictions(bd_preds, cfg['output_dir'], cfg)
    batdetect2_pipeline.delete_segments(segmented_file_paths)

    return bd_preds

def apply_pipeline_on_file(file, cfg):
    bd_preds = pd.DataFrame()

    if not cfg['output_dir'].is_dir():
        cfg['output_dir'].mkdir(parents=True, exist_ok=True)
    if not cfg['tmp_dir'].is_dir():
        cfg['tmp_dir'].mkdir(parents=True, exist_ok=True)

    cfg["csv_filename"] = f"batdetect2_pipeline_{file.name.split('.')[0]}"
    print(f"Generating detections for {file.name}")
    segmented_file_paths = batdetect2_pipeline.generate_segmented_paths([file], cfg)
    file_path_mappings = batdetect2_pipeline.initialize_mappings(segmented_file_paths, cfg)
    bd_preds = apply_models(file_path_mappings, cfg)
    if cfg['save']:
        batdetect2_pipeline._save_predictions(bd_preds, cfg['output_dir'], cfg)
    batdetect2_pipeline.delete_segments(segmented_file_paths)

    return bd_preds

In [7]:
input_file = Path('../../Downloads/recover-20220728/Carp/20220728_080000.WAV')

cfg = get_config()
cfg['segment_duration'] = 30.0
cfg['input_audio'] = input_file
cfg['tmp_dir'] = Path('../output')
cfg['output_dir'] = Path('../output_dir')
cfg['run_model'] = True
cfg['num_processes'] = 4
cfg['should_csv'] = False
cfg['save'] = True

# apply_pipeline_on_file(input_file, cfg)
print(f"Generating detections for {input_file.name}")
segmented_file_paths = batdetect2_pipeline.generate_segmented_paths([input_file], cfg)
file_path_mappings = batdetect2_pipeline.initialize_mappings(segmented_file_paths, cfg)

Generating detections for 20220728_080000.WAV


In [8]:
sequential_dets = run_models(file_path_mappings[:])
sequential_dets

100%|██████████| 60/60 [03:07<00:00,  3.13s/it]


,start_time,end_time,low_freq,high_freq,class,class_prob,det_prob,individual,event,input_file
0,0.0135,0.0219,36640,49331,Pipistrellus nathusii,0.539,0.588,-1,Echolocation,../../Downloads/recover-20220728/Carp/20220728...
1,0.1295,0.1389,37500,48700,Pipistrellus nathusii,0.601,0.626,-1,Echolocation,../../Downloads/recover-20220728/Carp/20220728...
2,0.2405,0.2484,36640,45200,Pipistrellus nathusii,0.569,0.639,-1,Echolocation,../../Downloads/recover-20220728/Carp/20220728...
3,0.3425,0.3520,36640,49395,Pipistrellus nathusii,0.574,0.613,-1,Echolocation,../../Downloads/recover-20220728/Carp/20220728...
4,0.4485,0.4569,37500,48215,Pipistrellus nathusii,0.651,0.667,-1,Echolocation,../../Downloads/recover-20220728/Carp/20220728...
...,...,...,...,...,...,...,...,...,...,...
86,1794.1435,1794.1580,22890,29234,Nyctalus leisleri,0.488,0.648,-1,Echolocation,../../Downloads/recover-20220728/Carp/20220728...
87,1794.3755,1794.3911,22890,27858,Nyctalus leisleri,0.565,0.629,-1,Echolocation,../../Downloads/recover-20220728/Carp/20220728...
88,1794.6075,1794.6260,22890,27947,Nyctalus leisleri,0.361,0.540,-1,Echolocation,../../Downloads/recover-20220728/Carp/20220728...
89,1794.7165,1794.7290,25468,29839,Nyctalus leisleri,0.611,0.676,-1,Echolocation,../../Downloads/recover-20220728/Carp/20220728...


In [9]:
new_loaded_fp_mappings = []
for i in range(len(file_path_mappings)):
    new_loaded_fp_mapping = file_path_mappings[i]

    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if os.path.isfile(new_loaded_fp_mapping['model'].model_path):
        NET_PARAMS = torch.load(new_loaded_fp_mapping['model'].model_path, map_location=DEVICE)
        
    if NET_PARAMS['params']['model_name'] == 'Net2DFast':
        MODEL = models.Net2DFast(NET_PARAMS['params']['num_filters'], num_classes=len(NET_PARAMS['params']['class_names']),
                                    emb_dim=NET_PARAMS['params']['emb_dim'], ip_height=NET_PARAMS['params']['ip_height'],
                                    resize_factor=NET_PARAMS['params']['resize_factor'])
    elif NET_PARAMS['params']['model_name'] == 'Net2DFastNoAttn':
        MODEL = models.Net2DFastNoAttn(NET_PARAMS['params']['num_filters'], num_classes=len(NET_PARAMS['params']['class_names']),
                                    emb_dim=NET_PARAMS['params']['emb_dim'], ip_height=NET_PARAMS['params']['ip_height'],
                                    resize_factor=NET_PARAMS['params']['resize_factor'])
    elif NET_PARAMS['params']['model_name'] == 'Net2DFastNoCoordConv':
        MODEL = models.Net2DFastNoCoordConv(NET_PARAMS['params']['num_filters'], num_classes=len(NET_PARAMS['params']['class_names']),
                                    emb_dim=NET_PARAMS['params']['emb_dim'], ip_height=NET_PARAMS['params']['ip_height'],
                                    resize_factor=NET_PARAMS['params']['resize_factor'])
    else:
        print('Error: unknown model.')
    MODEL.load_state_dict(NET_PARAMS['state_dict'])

    params = NET_PARAMS['params']
    params['device'] = DEVICE
    model = MODEL.to(params['device'])
    model.eval()

    new_loaded_fp_mapping['loaded_model'] = model
    new_loaded_fp_mapping['params'] = params

    new_loaded_fp_mappings += [new_loaded_fp_mapping]

In [10]:
new_loaded_fp_mappings

[{'audio_seg': {'input_filepath': PosixPath('../../Downloads/recover-20220728/Carp/20220728_080000.WAV'),
   'audio_file': PosixPath('../output/20220728_080000__0.00_30.00.wav'),
   'offset': 0.0},
  'model': <models.bat_call_detector.model_detector.BatCallDetector at 0x78a312ba7400>,
  'original_file_name': PosixPath('../../Downloads/recover-20220728/Carp/20220728_080000.WAV'),
  'loaded_model': Net2DFast(
    (conv_dn_0): ConvBlockDownCoordF(
      (conv): Conv2d(2, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (conv_bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (conv_dn_1): ConvBlockDownCoordF(
      (conv): Conv2d(33, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (conv_bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (conv_dn_2): ConvBlockDownCoordF(
      (conv): Conv2d(65, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (conv_bn): BatchNorm2d(

In [11]:
torch.set_num_threads(1)

In [12]:
def process_file(audio_file, model, params, args, time_exp=None, top_n=5, return_raw_preds=False, max_duration=False):

    # store temporary results here
    predictions = []
    spec_feats  = []
    cnn_feats   = []
    spec_slices = []

    # get time expansion  factor
    if time_exp is None:
        time_exp = args['time_expansion_factor']

    params['detection_threshold'] = args['detection_threshold']

    # load audio file
    sampling_rate, audio_full = au.load_audio_file(audio_file, time_exp,
                                   params['target_samp_rate'], params['scale_raw_audio'])
    
    duration_full = audio_full.shape[0] / float(sampling_rate)

    return_np_spec = args['spec_features'] or args['spec_slices']

    # loop through larger file and split into chunks
    # TODO fix so that it overlaps correctly and takes care of duplicate detections at borders
    num_chunks = int(np.ceil(duration_full/args['chunk_size']))
    for chunk_id in range(num_chunks):

        # chunk
        chunk_time   = args['chunk_size']*chunk_id
        chunk_length = int(sampling_rate*args['chunk_size'])
        start_sample = chunk_id*chunk_length
        end_sample   = np.minimum((chunk_id+1)*chunk_length, audio_full.shape[0])
        audio = audio_full[start_sample:end_sample]

        # load audio file and compute spectrogram
        duration, spec, spec_np = du.compute_spectrogram(audio, sampling_rate, params, return_np_spec)

        # evaluate model
        with torch.no_grad():
            outputs = model(spec, return_feats=args['cnn_features'])       

        # run non-max suppression
        pred_nms, features = pp.run_nms(outputs, params, np.array([float(sampling_rate)]))
        pred_nms = pred_nms[0]
        pred_nms['start_times'] += chunk_time
        pred_nms['end_times'] += chunk_time

        # if we have a background class
        if pred_nms['class_probs'].shape[0] > len(params['class_names']):
            pred_nms['class_probs'] = pred_nms['class_probs'][:-1, :]

        predictions.append(pred_nms)

        # extract features - if there are any calls detected
        if (pred_nms['det_probs'].shape[0] > 0):
            if args['spec_features']:
                spec_feats.append(feats.get_feats(spec_np, pred_nms, params))

            if args['cnn_features']:
                cnn_feats.append(features[0])

            if args['spec_slices']:
                spec_slices.extend(feats.extract_spec_slices(spec_np, pred_nms, params))

    # convert the predictions into output dictionary
    file_id = os.path.basename(audio_file)
    predictions, spec_feats, cnn_feats, spec_slices =\
              du.merge_results(predictions, spec_feats, cnn_feats, spec_slices)
    results = du.convert_results(file_id, time_exp, duration_full, params,
                              predictions, spec_feats, cnn_feats, spec_slices)

    # summarize results
    if not args['quiet']:
        num_detections = len(results['pred_dict']['annotation'])
        print('{}'.format(num_detections) + ' call(s) detected above the threshold.')

    # print results for top n classes
    if not args['quiet'] and (num_detections > 0):
        class_overall = pp.overall_class_pred(predictions['det_probs'], predictions['class_probs'])
        print('species name'.ljust(30) + 'probablity present')
        for cc in np.argsort(class_overall)[::-1][:top_n]:
            print(params['class_names'][cc].ljust(30) + str(round(class_overall[cc], 3)))

    if return_raw_preds:
        return predictions
    else:
        return results

In [13]:
def _run_batdetect(model_obj, audio_file, model, params): #
    """
    Parameters:: 
        audio_file: a path containing the post-processed wav file.

    Returns:: a pd.Dataframe containing the bat calls detections
    """

    # Suppress output from this call
    text_trap = io.StringIO()
    sys.stdout = text_trap

    model_output = process_file(
        audio_file=audio_file,
        model=model,
        params=params,
        args= {
            'detection_threshold': model_obj.detection_threshold,
            'spec_slices': model_obj.spec_slices,
            'chunk_size': model_obj.chunk_size,
            'quiet': model_obj.quiet,
            'spec_features' : False,
            'cnn_features': model_obj.cnn_features,
        },
        time_exp=model_obj.time_expansion_factor,
    )
    annotations = model_output['pred_dict']['annotation']

    # Restore stdout
    sys.stdout = sys.__stdout__

    out_df = gen_empty_df()
    if annotations:
        out_df = pd.DataFrame.from_records(annotations) 
        # out_df['detection_confidence'] = out_df['det_prob']
        # out_df.drop(columns = ['class', 'class_prob', 'det_prob','individual'], inplace=True)
    return out_df


def apply_model(file_mapping):
    """
    Runs the batdetect2 model on a single provided audio segmens and corrects the offsets according the segment.

    Parameters
    ------------
    file_mappings : `List`
        - List of dictionaries generated by initialize_mappings()

    Returns
    ------------
    corrected_bd_dets : `pandas.DataFrame`
        - A DataFrame of detections that will also be saved in the provided output_dir under the above csv_name
        - 7 columns in this DataFrame: start_time, end_time, low_freq, high_freq, detection_confidence, event, input_file
        - Detections are always specified w.r.t their input_file; earliest start_time can be 0 and latest end_time can be 1795.
        - Events are always "Echolocation" as we are using a model that only detects search-phase calls.
    """

    bd_dets = _run_batdetect(file_mapping['model'], file_mapping['audio_seg']['audio_file'], 
                             file_mapping['loaded_model'], file_mapping['params'])
    corrected_bd_dets = pipeline._correct_annotation_offsets(
                                                            bd_dets,
                                                            file_mapping['original_file_name'],
                                                            file_mapping['audio_seg']['offset']
                                                            )

    return corrected_bd_dets

In [14]:
input = new_loaded_fp_mappings[:]
num_processes = 16
pool = multiprocessing.Pool(processes=num_processes)
chunksize_custom = 1
print(f'Parsing {len(input)} chunks with {num_processes} processors and {chunksize_custom} chunks per processor')
start = time.time()
results = tqdm(pool.imap(apply_model, input, chunksize=chunksize_custom), 
                        desc=f"Applying BatDetect2", total=len(input),)
parallel_dets = gen_empty_df() 
parallel_dets = pd.concat(results, ignore_index=True)
end = time.time()
parallel_dets

Applying BatDetect2: 100%|██████████| 60/60 [00:53<00:00,  1.12it/s]


Parsing 60 chunks with 16 processors and 1 chunks per processor


,start_time,end_time,low_freq,high_freq,class,class_prob,det_prob,individual,event,input_file
0,0.0135,0.0219,36640,49331,Pipistrellus nathusii,0.539,0.588,-1,Echolocation,../../Downloads/recover-20220728/Carp/20220728...
1,0.1295,0.1389,37500,48700,Pipistrellus nathusii,0.601,0.626,-1,Echolocation,../../Downloads/recover-20220728/Carp/20220728...
2,0.2405,0.2484,36640,45200,Pipistrellus nathusii,0.569,0.639,-1,Echolocation,../../Downloads/recover-20220728/Carp/20220728...
3,0.3425,0.3520,36640,49395,Pipistrellus nathusii,0.574,0.613,-1,Echolocation,../../Downloads/recover-20220728/Carp/20220728...
4,0.4485,0.4569,37500,48215,Pipistrellus nathusii,0.651,0.667,-1,Echolocation,../../Downloads/recover-20220728/Carp/20220728...
...,...,...,...,...,...,...,...,...,...,...
4460,1794.1435,1794.1580,22890,29234,Nyctalus leisleri,0.488,0.648,-1,Echolocation,../../Downloads/recover-20220728/Carp/20220728...
4461,1794.3755,1794.3911,22890,27858,Nyctalus leisleri,0.565,0.629,-1,Echolocation,../../Downloads/recover-20220728/Carp/20220728...
4462,1794.6075,1794.6260,22890,27947,Nyctalus leisleri,0.361,0.540,-1,Echolocation,../../Downloads/recover-20220728/Carp/20220728...
4463,1794.7165,1794.7290,25468,29839,Nyctalus leisleri,0.611,0.676,-1,Echolocation,../../Downloads/recover-20220728/Carp/20220728...
